# 08 — QC-Analyse: Rating-Delta nach ELO-Klasse und Zeitraum

Für jeden Spieler werden die QC-Fenster ab Jan 2015 ausgewertet:
```
delta = expected_change (TXT-Snapshot) − scraped_change (Σ rating_change_weighted)
```
**ELO-Klassifikation** basiert auf dem Rating vom **Jan 2023** (vor der FIDE-Korrektur für Spieler unter 2000).

Jede Pivot-Tabelle zeigt den **Prozentanteil** der Fenster pro Delta-Bucket für jeden der 23 Halbjahreszeiträume (Feb 2015 → Jan 2026).

In [17]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _setup import load_query, apply_style

import numpy as np
import pandas as pd
from IPython.display import display

apply_style()


## Datenbasis

In [18]:
sql = '''
WITH jan2023 AS (
    -- Letzte verfügbare Rating-Eintrag auf oder vor Jan 2023 pro Spieler
    SELECT DISTINCT ON (fide_id)
        fide_id,
        COALESCE(std_rating, published_rating) AS rating_2023
    FROM rating_history
    WHERE period <= '2023-01-01'
      AND COALESCE(std_rating, published_rating) IS NOT NULL
    ORDER BY fide_id, period DESC
)
SELECT
    q.fide_id,
    p.name,
    COALESCE(j.rating_2023, p.std_rating)               AS rating_2023,
    CASE
        WHEN COALESCE(j.rating_2023, p.std_rating) >= 2400 THEN 'ELO >= 2400'
        WHEN COALESCE(j.rating_2023, p.std_rating) >= 2000 THEN 'ELO 2000-2400'
        ELSE                                                     'ELO < 2000'
    END                                                  AS elo_group,
    TO_CHAR(q.period_start, 'YY-MM')                    AS period,
    q.period_start,
    q.expected_change,
    q.scraped_change,
    q.delta,
    q.flag
FROM qc_rating_check q
JOIN players p USING (fide_id)
LEFT JOIN jan2023 j USING (fide_id)
WHERE q.period_start >= '2015-01-01'
ORDER BY q.fide_id, q.period_start
'''

df = load_query(sql)
df['period_start'] = pd.to_datetime(df['period_start'])
df['abs_delta'] = df['delta'].abs()

# Feste Perioden-Reihenfolge (alle 23 Halbjahreszeiträume ab 2015)
ALL_PERIODS = sorted(df['period'].unique())

print(f"{len(df):,} QC-Fenster geladen, {df['fide_id'].nunique():,} Spieler")
print(f"ELO-Gruppen (Basis Jan 2023):")
print(df.groupby('elo_group')['fide_id'].nunique().rename('Spieler'))


Task was destroyed but it is pending!
task: <Task pending name='Task-125' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/macminipit/Projekte/fide-scraper/.venv/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-126' coro=<Kernel.shell_main() running at /Users/macminipit/Projekte/fide-scraper/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/macminipit/Projekte/fide-scraper/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/macminipit/Projekte/fide-scraper/.venv/lib/python3.13/site-packages/pandas/io/sql.py:2796: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  result = cur.fetchall()
Task was destroyed but it is pending!
task: <Task pending name='Task-126' coro=<Kernel.shell_main() running at /Users/macminipit/Projekte/fide-scraper/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.

24,149 QC-Fenster geladen, 1,092 Spieler
ELO-Gruppen (Basis Jan 2023):
elo_group
ELO 2000-2400    297
ELO < 2000       105
ELO >= 2400      690
Name: Spieler, dtype: int64


## Hilfsfunktionen

In [19]:
BUCKET_BINS   = [-np.inf, 0.001, 2, 5, 10, 20, 50, np.inf]
BUCKET_LABELS = ['=0', '1-2', '3-5', '6-10', '11-20', '21-50', '>50']
OK_CUTOFF = 5   # |delta| <= 5 gilt als ok

def assign_bucket(s):
    abs_s = s.abs()
    return pd.cut(abs_s, bins=BUCKET_BINS, labels=BUCKET_LABELS, right=True)

def pivot_table(sub: pd.DataFrame, title: str) -> pd.DataFrame:
    sub = sub.copy()
    sub['bucket'] = assign_bucket(sub['delta'])

    # Absolute Counts
    counts = (
        sub.groupby(['bucket', 'period'], observed=True)
           .size()
           .unstack('period', fill_value=0)
           .reindex(columns=ALL_PERIODS, fill_value=0)
           .reindex(BUCKET_LABELS)
    )
    # Periode-Totals
    totals = counts.sum(axis=0)

    # Prozentwerte pro Spalte
    pct = counts.div(totals, axis=1).mul(100).round(1)

    # OK-Summe einfügen (nach Zeile '3-5', vor '6-10')
    ok_row = pct.loc[['=0', '1-2', '3-5']].sum(axis=0).round(1).rename('--- ok ≤5 ---')
    pct = pd.concat([pct.iloc[:3], ok_row.to_frame().T, pct.iloc[3:]])

    # N pro Periode als letzte Zeile
    n_row = totals.rename('N (Fenster)')
    pct = pd.concat([pct, n_row.to_frame().T])

    pct.index.name = f'|delta|   {title}'
    return pct

def style_pivot(pct: pd.DataFrame) -> 'pd.io.formats.style.Styler':
    """Farbliche Hervorhebung: Zellen im schlechten Bereich rot, im guten grün."""
    def color(val, row_label):
        if row_label in ('--- ok ≤5 ---',):
            # Grün-Skala: je höher desto grüner
            try:
                v = float(val)
                intensity = int(min(v, 100) / 100 * 120)
                return f'background-color: rgb({255-intensity},{255},{255-intensity}); color: black'
            except Exception:
                return ''
        if row_label in ('>50', '21-50'):
            try:
                v = float(val)
                if v == 0:
                    return ''
                intensity = int(min(v * 4, 100) / 100 * 150)
                return f'background-color: rgb(255,{255-intensity},{255-intensity}); color: black'
            except Exception:
                return ''
        return ''

    styled = pct.style
    for row_label in pct.index:
        styled = styled.apply(
            lambda col, rl=row_label: [color(v, rl) for v in col],
            subset=pd.IndexSlice[row_label, :],
            axis=1
        )
    styled = styled.format(
        lambda v: f'{v:.1f}' if isinstance(v, float) else str(int(v)),
        na_rep='-'
    )
    styled = styled.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '10px'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('font-size', '10px'), ('text-align', 'right'), ('padding', '2px 6px')]},
        {'selector': 'tr:nth-child(4)', 'props': [('border-top', '2px solid #555'), ('font-weight', 'bold')]},
    ])
    return styled


In [20]:
def top10_table(sub: pd.DataFrame, title: str) -> pd.DataFrame:
    """Top-10 Spieler mit der grössten kumulativen absoluten ELO-Abweichung.
    Ausgabe: Spieler in Zeilen, Zeiträume in Spalten (delta-Wert pro Zelle).
    Letzte drei Spalten: Σ|Δ|, Ø|Δ|, Max|Δ|.
    """
    # Rang nach Σ|Δ|
    ranking = (
        sub.groupby(['fide_id', 'name', 'rating_2023'])['abs_delta']
           .sum()
           .sort_values(ascending=False)
           .head(10)
           .reset_index()
    )
    top_ids = ranking['fide_id'].tolist()

    # Pivot: Spieler × Periode, Werte = delta (mit Vorzeichen)
    top_df = sub[sub['fide_id'].isin(top_ids)].copy()
    pivot = (
        top_df.pivot_table(index='fide_id', columns='period', values='delta',
                           aggfunc='first')
              .reindex(columns=ALL_PERIODS)
    )

    # Summenspalten anhängen
    pivot['Σ|Δ|']  = top_df.groupby('fide_id')['abs_delta'].sum().round(1)
    pivot['Ø|Δ|']  = top_df.groupby('fide_id')['abs_delta'].mean().round(1)
    pivot['Max|Δ|'] = top_df.groupby('fide_id')['abs_delta'].max().round(1)

    # Spieler-Labels als Index (Name + ELO)
    label_map = ranking.set_index('fide_id').apply(
        lambda r: f"{r['name']} ({int(r['rating_2023']) if pd.notna(r['rating_2023']) else '?'})", axis=1
    )
    pivot.index = pivot.index.map(label_map)
    pivot = pivot.reindex(ranking['fide_id'].map(label_map))   # Rangordnung

    pivot.index.name = 'Spieler (ELO Jan23)'
    pivot.columns.name = None

    # Styling: positive delta = rot (zu wenig scraped), negativ = blau
    SUMMARY_COLS = ['Σ|Δ|', 'Ø|Δ|', 'Max|Δ|']
    period_cols  = [c for c in pivot.columns if c not in SUMMARY_COLS]

    def color_cell(v):
        try:
            f = float(v)
        except (TypeError, ValueError):
            return ''
        if np.isnan(f):
            return ''
        if abs(f) <= 5:
            return 'color: #888'
        if f > 0:
            intensity = int(min(abs(f) / 50 * 200, 200))
            return f'background-color: rgb(255,{255-intensity},{255-intensity})'
        else:
            intensity = int(min(abs(f) / 50 * 200, 200))
            return f'background-color: rgb({255-intensity},{255-intensity},255)'

    styled = (
        pivot.style
             .map(color_cell, subset=period_cols)
             .format(lambda v: f'{v:+.1f}' if isinstance(v, float) and not np.isnan(v) else '',
                     subset=period_cols)
             .format('{:.1f}', subset=SUMMARY_COLS, na_rep='-')
             .set_table_styles([
                 {'selector': 'th', 'props': [('font-size', '10px'), ('text-align', 'center')]},
                 {'selector': 'td', 'props': [('font-size', '10px'), ('text-align', 'right'),
                                              ('padding', '2px 5px'), ('min-width', '38px')]},
             ])
    )

    print(f'\n--- {title}: Top-10 nach Σ|Δ| (delta = erwartet − gescraped, +rot / -blau) ---')
    display(styled)
    return pivot


---
## 1 — Gesamt

In [21]:
from IPython.display import HTML
print('=' * 70)
print('GESAMT — alle Spieler in qc_rating_check ab 2015')
print('=' * 70)
pct_all = pivot_table(df, 'Gesamt')
display(style_pivot(pct_all))
top10_table(df, 'Gesamt')


GESAMT — alle Spieler in qc_rating_check ab 2015


period,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,20-01,20-07,21-01,21-07,22-01,22-07,23-01,23-07,24-01,24-07,25-01,25-07,26-01
|delta| Gesamt,,,,,,,,,,,,,,,,,,,,,,,
=0,14.3,15.3,13.1,14.9,14.5,15.9,16.3,16.4,15.5,16.2,25.9,62.2,64.1,28.0,23.9,20.6,19.7,19.7,18.4,21.2,18.8,20.1,23.2
1-2,23.7,22.2,25.8,25.3,24.3,23.5,28.0,26.9,29.7,28.0,29.5,27.0,16.9,38.4,27.9,28.6,28.5,27.1,25.3,28.1,29.2,27.8,19.6
3-5,11.9,14.8,14.3,13.4,13.3,14.7,13.8,13.8,12.0,12.4,12.0,2.4,3.5,8.2,9.7,12.4,12.5,13.6,15.9,14.7,13.9,15.8,15.8
--- ok ≤5 ---,49.9,52.3,53.2,53.6,52.1,54.1,58.1,57.1,57.2,56.6,67.4,91.6,84.5,74.6,61.5,61.6,60.7,60.4,59.6,64.0,61.9,63.7,58.6
6-10,14.7,15.6,16.7,16.5,17.5,16.7,16.5,16.2,14.8,15.0,15.5,3.2,6.4,10.1,13.3,15.2,15.9,15.8,15.2,16.6,15.6,14.5,17.3
11-20,13.9,16.8,15.7,15.7,15.0,16.2,14.0,13.8,14.1,14.9,9.8,2.2,4.5,9.4,15.0,12.5,12.5,14.7,13.3,13.2,14.3,13.9,15.4
21-50,14.7,10.3,9.6,10.2,10.8,8.8,7.6,8.1,9.0,9.0,5.6,2.1,3.1,4.3,7.4,7.4,9.4,7.8,7.3,5.3,6.6,6.6,8.0
>50,6.8,5.1,4.9,4.1,4.5,4.3,3.9,4.7,5.0,4.5,1.8,0.8,1.6,1.7,2.9,3.4,1.4,1.5,4.6,0.9,1.6,1.4,0.7
N (Fenster),958.0,969.0,989.0,1001.0,1012.0,1026.0,1038.0,1043.0,1050.0,1054.0,1062.0,1062.0,1067.0,1069.0,1070.0,1074.0,1080.0,1083.0,1083.0,1086.0,1089.0,1092.0,1092.0



--- Gesamt: Top-10 nach Σ|Δ| (delta = erwartet − gescraped, +rot / -blau) ---


,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,20-01,20-07,21-01,21-07,22-01,22-07,23-01,23-07,24-01,24-07,25-01,25-07,26-01,Σ|Δ|,Ø|Δ|,Max|Δ|
Spieler (ELO Jan23),,,,,,,,,,,,,,,,,,,,,,,,,,
Garv Rai (2181),+47.8,-45.6,-21.6,+20.8,+418.4,-418.4,-54.6,-1.4,-2.0,+181.6,-124.3,+0.0,+0.0,-25.0,+45.0,-19.6,-40.2,+39.8,+0.4,-27.0,+27.4,+0.3,+0.0,1561.0,67.9,418.4
Harsh Suresh (2280),-66.2,-24.8,-0.2,-0.2,+129.8,-204.4,+118.5,-43.8,+142.7,-140.4,-1.8,+0.0,+0.0,+156.4,-226.4,+52.0,+103.6,-91.9,-1.1,+5.3,+4.0,+5.0,-3.7,1522.2,66.2,226.4
Mayank Chakraborty (2179),,,,,,-143.6,+28.6,-39.3,+72.9,-52.5,-5.8,+0.0,+0.4,+0.5,+90.2,-206.8,+300.5,-179.4,-7.1,-13.8,+6.1,+11.5,+26.7,1185.7,65.9,300.5
"Pasztor, Balazs (2400)",+55.0,-77.2,+121.8,-103.4,-17.4,+6.6,-76.6,+76.6,-83.4,+134.2,-58.8,-26.2,+96.6,-82.0,+13.2,+2.2,-27.1,+48.3,-31.1,+7.8,-0.2,+2.8,-7.2,1155.7,50.2,134.2
"Bodrogi, Bendeguz (2073)",,,+78.2,-221.6,+77.0,+0.0,+91.2,-45.6,-31.2,+10.2,-25.8,+0.2,+105.6,-142.9,+0.4,+70.5,+66.4,-117.8,+28.4,-13.4,+2.7,-1.4,-1.9,1132.4,53.9,221.6
"Piliposyan, Robert (2295)",-0.2,+48.4,-48.2,+47.6,-46.2,+14.7,-40.0,+86.0,-92.6,-30.2,+62.8,-38.0,-10.0,+48.3,+163.8,-162.0,-0.4,-0.6,+1.2,-23.1,+21.6,-21.5,+22.0,1029.4,44.8,163.8
"Collin, Moritz Valentin (2123)",,,,,,,,,+148.6,-94.4,-93.8,+74.4,+17.6,-18.8,+142.5,-163.9,+36.8,-17.0,-56.9,+75.2,-23.4,+32.6,+24.2,1020.2,68.0,163.9
"Jacobson, Brandon (2551)",-34.6,-155.0,+67.4,+29.8,+169.8,-123.2,+94.2,-83.6,-39.9,+27.6,+1.3,+4.0,+8.6,-8.5,+13.9,-22.7,-7.0,+42.3,-31.1,-12.6,+0.0,+14.6,-9.3,1001.0,43.5,169.8
"Davtyan, Arsen (2293)",,,+38.4,-5.0,+58.2,-107.9,+114.2,-4.6,-15.6,-168.4,+121.8,+38.0,+33.8,-71.8,+62.2,-64.0,+0.2,+0.2,-5.6,-19.5,+43.2,-6.6,-12.0,991.2,47.2,168.4


,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,...,23-01,23-07,24-01,24-07,25-01,25-07,26-01,Σ|Δ|,Ø|Δ|,Max|Δ|
Spieler (ELO Jan23),,,,,,,,,,,,,,,,,,,,,
Garv Rai (2181),47.8,-45.6,-21.6,20.8,418.4,-418.40,-54.6,-1.38,-1.96,181.55,...,-40.20,39.78,0.36,-27.0,27.36,0.32,0.0,1561.0,67.9,418.4
Harsh Suresh (2280),-66.2,-24.8,-0.2,-0.2,129.8,-204.40,118.5,-43.80,142.66,-140.40,...,103.60,-91.93,-1.10,5.3,4.00,5.00,-3.7,1522.2,66.2,226.4
Mayank Chakraborty (2179),NaN,NaN,NaN,NaN,NaN,-143.60,28.6,-39.28,72.88,-52.45,...,300.52,-179.40,-7.10,-13.8,6.10,11.50,26.7,1185.7,65.9,300.5
"Pasztor, Balazs (2400)",55.0,-77.2,121.8,-103.4,-17.4,6.60,-76.6,76.60,-83.40,134.24,...,-27.10,48.30,-31.10,7.8,-0.20,2.80,-7.2,1155.7,50.2,134.2
"Bodrogi, Bendeguz (2073)",NaN,NaN,78.2,-221.6,77.0,0.00,91.2,-45.60,-31.20,10.18,...,66.40,-117.80,28.40,-13.4,2.70,-1.40,-1.9,1132.4,53.9,221.6
"Piliposyan, Robert (2295)",-0.2,48.4,-48.2,47.6,-46.2,14.72,-40.0,86.00,-92.60,-30.20,...,-0.40,-0.60,1.20,-23.1,21.60,-21.50,22.0,1029.4,44.8,163.8
"Collin, Moritz Valentin (2123)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.60,-94.40,...,36.80,-17.04,-56.92,75.2,-23.40,32.60,24.2,1020.2,68.0,163.9
"Jacobson, Brandon (2551)",-34.6,-155.0,67.4,29.8,169.8,-123.20,94.2,-83.60,-39.90,27.60,...,-7.00,42.30,-31.10,-12.6,0.00,14.60,-9.3,1001.0,43.5,169.8
"Davtyan, Arsen (2293)",NaN,NaN,38.4,-5.0,58.2,-107.86,114.2,-4.60,-15.60,-168.45,...,0.20,0.20,-5.60,-19.5,43.20,-6.60,-12.0,991.2,47.2,168.4


---
## 2 — ELO ≥ 2400

In [22]:
print('=' * 70)
print('ELO >= 2400  (gemessen Jan 2023)')
print('=' * 70)
sub = df[df['elo_group'] == 'ELO >= 2400']
print(f'{sub["fide_id"].nunique()} Spieler, {len(sub):,} Fenster')
pct_2400 = pivot_table(sub, 'ELO >= 2400')
display(style_pivot(pct_2400))
top10_table(sub, 'ELO >= 2400')


ELO >= 2400  (gemessen Jan 2023)
690 Spieler, 15,826 Fenster


period,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,20-01,20-07,21-01,21-07,22-01,22-07,23-01,23-07,24-01,24-07,25-01,25-07,26-01
|delta| ELO >= 2400,,,,,,,,,,,,,,,,,,,,,,,
=0,14.7,15.2,13.3,14.7,13.7,14.2,16.3,16.4,14.8,17.4,26.4,60.9,58.6,30.1,26.2,23.5,23.0,23.8,24.6,25.9,25.1,26.8,31.4
1-2,25.5,23.6,25.9,25.1,25.0,25.0,28.0,28.3,31.2,28.4,30.0,28.4,20.7,38.8,30.6,29.1,31.3,29.6,26.8,27.5,31.0,28.4,22.3
3-5,12.4,14.6,13.9,13.8,12.7,15.7,15.1,13.6,14.1,14.3,14.1,3.5,4.9,9.4,9.6,12.5,12.9,14.5,16.8,16.1,13.9,15.9,16.4
--- ok ≤5 ---,52.6,53.4,53.1,53.6,51.4,54.9,59.4,58.3,60.1,60.1,70.5,92.8,84.2,78.3,66.4,65.1,67.2,67.9,68.2,69.5,70.0,71.1,70.1
6-10,14.7,15.3,17.3,17.8,19.5,18.0,17.9,18.6,16.7,15.7,15.5,3.6,7.8,10.0,14.3,17.0,15.4,14.5,16.5,15.2,13.9,12.6,14.1
11-20,12.4,16.1,15.5,15.6,15.7,16.0,14.5,13.8,14.8,15.7,9.7,2.3,5.2,8.6,14.2,12.2,10.7,12.3,11.3,12.0,12.2,12.3,11.4
21-50,13.5,9.6,8.9,9.3,10.2,8.3,6.0,6.7,6.5,7.1,3.6,1.2,2.2,2.5,4.8,5.1,6.7,5.4,3.9,3.2,3.9,3.9,4.3
>50,6.8,5.6,5.3,3.6,3.2,2.8,2.3,2.6,2.0,1.4,0.7,0.1,0.6,0.6,0.3,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
N (Fenster),675.0,678.0,684.0,686.0,687.0,688.0,689.0,689.0,690.0,690.0,690.0,690.0,690.0,690.0,690.0,690.0,690.0,690.0,690.0,690.0,690.0,690.0,690.0



--- ELO >= 2400: Top-10 nach Σ|Δ| (delta = erwartet − gescraped, +rot / -blau) ---


,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,20-01,20-07,21-01,21-07,22-01,22-07,23-01,23-07,24-01,24-07,25-01,25-07,26-01,Σ|Δ|,Ø|Δ|,Max|Δ|
Spieler (ELO Jan23),,,,,,,,,,,,,,,,,,,,,,,,,,
"Pasztor, Balazs (2400)",+55.0,-77.2,+121.8,-103.4,-17.4,+6.6,-76.6,+76.6,-83.4,+134.2,-58.8,-26.2,+96.6,-82.0,+13.2,+2.2,-27.1,+48.3,-31.1,+7.8,-0.2,+2.8,-7.2,1155.7,50.2,134.2
"Jacobson, Brandon (2551)",-34.6,-155.0,+67.4,+29.8,+169.8,-123.2,+94.2,-83.6,-39.9,+27.6,+1.3,+4.0,+8.6,-8.5,+13.9,-22.7,-7.0,+42.3,-31.1,-12.6,+0.0,+14.6,-9.3,1001.0,43.5,169.8
"Baum, Jonasz (2420)",-182.6,-24.4,-0.4,+0.8,+144.8,-145.2,+42.8,-40.9,+12.6,-58.2,+46.6,-59.6,+60.0,+0.8,+9.3,-10.1,-7.5,+5.2,+14.4,-11.1,-5.3,+5.6,-4.8,893.0,38.8,182.6
"Kucuksari, Kaan (2492)",+64.2,+101.4,-164.8,+77.4,-76.0,-53.2,+49.4,+23.4,-12.6,-39.4,+28.6,+0.4,+0.0,+25.2,-24.6,-5.6,+7.1,-0.2,+0.0,+6.6,-6.6,-5.7,-1.1,773.5,33.6,164.8
Aditya Mittal (2486),-49.2,+19.6,+83.0,-108.2,-7.4,-99.2,+113.7,-6.8,-32.7,+31.3,+0.3,+0.0,+0.0,+23.9,-29.6,+44.8,-28.8,-19.0,-5.0,+11.5,+6.5,-21.5,+14.0,756.0,32.9,113.7
"Bournel, Antoine (2404)",+16.4,+29.6,-31.0,-19.8,+34.8,-115.1,+99.6,+56.6,-65.8,-13.4,+24.8,-0.6,+0.0,+35.0,-63.0,-3.9,+35.2,-13.0,+20.8,-10.9,+11.4,-10.4,+3.1,714.2,31.1,115.1
"Klimkowski, Jan (2447)",,,,-12.6,+241.8,-204.8,+2.2,-27.2,+52.8,-52.4,+0.1,-9.0,+8.8,+3.2,-17.6,+12.9,+11.2,-10.6,-1.4,+9.8,-5.9,-7.3,-19.4,711.1,35.6,241.8
"Lazavik, Denis (2529)",,,-84.4,+213.4,-196.4,+24.2,-24.2,+67.0,-32.2,-34.6,+0.2,+0.3,+1.2,+0.4,-0.6,-3.6,+5.0,+0.8,+0.4,+0.0,+0.4,+0.3,+0.0,689.6,32.8,213.4
"De Silva, L M S T (2452)",-9.2,+18.6,-8.8,+46.4,-45.0,-2.3,+4.0,+125.4,-114.4,+79.2,-91.0,-0.2,+0.0,+61.6,-5.3,-55.1,+0.1,+3.8,-3.5,+3.6,-3.9,-1.4,-5.3,688.1,29.9,125.4


,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,...,23-01,23-07,24-01,24-07,25-01,25-07,26-01,Σ|Δ|,Ø|Δ|,Max|Δ|
Spieler (ELO Jan23),,,,,,,,,,,,,,,,,,,,,
"Pasztor, Balazs (2400)",55.0,-77.2,121.8,-103.4,-17.4,6.60,-76.60,76.60,-83.40,134.24,...,-27.1,48.3,-31.1,7.8,-0.2,2.8,-7.2,1155.7,50.2,134.2
"Jacobson, Brandon (2551)",-34.6,-155.0,67.4,29.8,169.8,-123.20,94.20,-83.60,-39.90,27.60,...,-7.0,42.3,-31.1,-12.6,0.0,14.6,-9.3,1001.0,43.5,169.8
"Baum, Jonasz (2420)",-182.6,-24.4,-0.4,0.8,144.8,-145.20,42.80,-40.92,12.60,-58.20,...,-7.5,5.2,14.4,-11.1,-5.3,5.6,-4.8,893.0,38.8,182.6
"Kucuksari, Kaan (2492)",64.2,101.4,-164.8,77.4,-76.0,-53.20,49.40,23.40,-12.60,-39.40,...,7.1,-0.2,0.0,6.6,-6.6,-5.7,-1.1,773.5,33.6,164.8
Aditya Mittal (2486),-49.2,19.6,83.0,-108.2,-7.4,-99.20,113.68,-6.85,-32.70,31.30,...,-28.8,-19.0,-5.0,11.5,6.5,-21.5,14.0,756.0,32.9,113.7
"Bournel, Antoine (2404)",16.4,29.6,-31.0,-19.8,34.8,-115.08,99.60,56.62,-65.80,-13.40,...,35.2,-13.0,20.8,-10.9,11.4,-10.4,3.1,714.2,31.1,115.1
"Klimkowski, Jan (2447)",NaN,NaN,NaN,-12.6,241.8,-204.80,2.20,-27.25,52.80,-52.40,...,11.2,-10.6,-1.4,9.8,-5.9,-7.3,-19.4,711.1,35.6,241.8
"Lazavik, Denis (2529)",NaN,NaN,-84.4,213.4,-196.4,24.20,-24.20,67.00,-32.20,-34.60,...,5.0,0.8,0.4,0.0,0.4,0.3,0.0,689.6,32.8,213.4
"De Silva, L M S T (2452)",-9.2,18.6,-8.8,46.4,-45.0,-2.30,4.00,125.40,-114.42,79.17,...,0.1,3.8,-3.5,3.6,-3.9,-1.4,-5.3,688.1,29.9,125.4


---
## 3 — ELO 2000–2400

In [23]:
print('=' * 70)
print('ELO 2000–2400  (gemessen Jan 2023)')
print('=' * 70)
sub = df[df['elo_group'] == 'ELO 2000-2400']
print(f'{sub["fide_id"].nunique()} Spieler, {len(sub):,} Fenster')
pct_mid = pivot_table(sub, 'ELO 2000-2400')
display(style_pivot(pct_mid))
top10_table(sub, 'ELO 2000-2400')


ELO 2000–2400  (gemessen Jan 2023)
297 Spieler, 6,531 Fenster


period,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,20-01,20-07,21-01,21-07,22-01,22-07,23-01,23-07,24-01,24-07,25-01,25-07,26-01
|delta| ELO 2000-2400,,,,,,,,,,,,,,,,,,,,,,,
=0,11.8,12.4,9.4,11.1,13.0,16.8,11.8,13.5,13.7,11.3,23.5,66.6,73.4,20.8,18.1,13.7,12.3,10.2,9.8,10.8,7.4,8.4,9.1
1-2,20.2,18.2,25.6,26.8,23.4,19.6,26.5,22.5,28.5,27.4,27.6,22.2,9.2,37.9,24.6,30.0,25.6,23.4,29.5,27.4,26.3,28.3,15.2
3-5,10.5,16.1,16.1,12.6,16.4,14.3,12.2,14.9,7.9,8.6,8.2,0.7,1.0,7.2,11.9,13.7,11.9,13.6,16.3,13.5,15.2,17.2,15.5
--- ok ≤5 ---,42.5,46.7,51.1,50.5,52.8,50.7,50.5,50.9,50.1,47.3,59.3,89.5,83.6,65.9,54.6,57.4,49.8,47.2,55.6,51.7,48.9,53.9,39.8
6-10,16.0,16.9,16.5,14.9,15.2,15.4,15.7,12.5,12.0,15.1,16.0,1.7,3.4,9.2,11.6,11.9,17.7,19.7,16.3,19.3,19.2,18.2,25.9
11-20,16.4,20.2,16.1,16.5,11.9,16.8,14.6,14.9,13.7,15.8,10.6,1.4,3.8,12.3,16.7,14.3,16.4,19.3,16.9,18.2,20.5,17.8,20.9
21-50,18.1,12.0,11.4,12.3,11.9,8.6,11.1,11.1,12.7,12.0,9.6,4.8,5.1,8.9,11.6,9.9,13.0,10.8,9.5,8.8,8.8,9.1,11.8
>50,7.1,4.1,4.7,5.7,8.2,8.6,8.0,10.7,11.3,9.9,4.4,2.7,4.1,3.8,5.5,6.5,3.1,3.1,1.7,2.0,2.7,1.0,1.7
N (Fenster),238.0,242.0,254.0,261.0,269.0,280.0,287.0,289.0,291.0,292.0,293.0,293.0,293.0,293.0,293.0,293.0,293.0,295.0,295.0,296.0,297.0,297.0,297.0



--- ELO 2000-2400: Top-10 nach Σ|Δ| (delta = erwartet − gescraped, +rot / -blau) ---


,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,20-01,20-07,21-01,21-07,22-01,22-07,23-01,23-07,24-01,24-07,25-01,25-07,26-01,Σ|Δ|,Ø|Δ|,Max|Δ|
Spieler (ELO Jan23),,,,,,,,,,,,,,,,,,,,,,,,,,
Garv Rai (2181),+47.8,-45.6,-21.6,+20.8,+418.4,-418.4,-54.6,-1.4,-2.0,+181.6,-124.3,+0.0,+0.0,-25.0,+45.0,-19.6,-40.2,+39.8,+0.4,-27.0,+27.4,+0.3,+0.0,1561.0,67.9,418.4
Harsh Suresh (2280),-66.2,-24.8,-0.2,-0.2,+129.8,-204.4,+118.5,-43.8,+142.7,-140.4,-1.8,+0.0,+0.0,+156.4,-226.4,+52.0,+103.6,-91.9,-1.1,+5.3,+4.0,+5.0,-3.7,1522.2,66.2,226.4
Mayank Chakraborty (2179),,,,,,-143.6,+28.6,-39.3,+72.9,-52.5,-5.8,+0.0,+0.4,+0.5,+90.2,-206.8,+300.5,-179.4,-7.1,-13.8,+6.1,+11.5,+26.7,1185.7,65.9,300.5
"Bodrogi, Bendeguz (2073)",,,+78.2,-221.6,+77.0,+0.0,+91.2,-45.6,-31.2,+10.2,-25.8,+0.2,+105.6,-142.9,+0.4,+70.5,+66.4,-117.8,+28.4,-13.4,+2.7,-1.4,-1.9,1132.4,53.9,221.6
"Piliposyan, Robert (2295)",-0.2,+48.4,-48.2,+47.6,-46.2,+14.7,-40.0,+86.0,-92.6,-30.2,+62.8,-38.0,-10.0,+48.3,+163.8,-162.0,-0.4,-0.6,+1.2,-23.1,+21.6,-21.5,+22.0,1029.4,44.8,163.8
"Collin, Moritz Valentin (2123)",,,,,,,,,+148.6,-94.4,-93.8,+74.4,+17.6,-18.8,+142.5,-163.9,+36.8,-17.0,-56.9,+75.2,-23.4,+32.6,+24.2,1020.2,68.0,163.9
"Davtyan, Arsen (2293)",,,+38.4,-5.0,+58.2,-107.9,+114.2,-4.6,-15.6,-168.4,+121.8,+38.0,+33.8,-71.8,+62.2,-64.0,+0.2,+0.2,-5.6,-19.5,+43.2,-6.6,-12.0,991.2,47.2,168.4
Aarav Dengla (2357),,,,+0.0,+0.0,+1.4,-91.4,+93.6,-0.9,-66.2,+63.6,+0.0,+0.0,+32.5,+167.6,-178.9,-19.6,+0.6,-14.0,+7.8,+120.4,-113.7,-0.1,972.4,48.6,178.9
"Ljepic, Andrej (2131)",,,,,+30.2,+11.2,+188.8,-188.2,-71.4,+71.1,-6.0,+80.2,-66.6,+43.5,-43.3,-7.6,+6.8,-6.4,-0.2,+8.4,-45.0,+37.7,+14.6,927.2,48.8,188.8


,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,...,23-01,23-07,24-01,24-07,25-01,25-07,26-01,Σ|Δ|,Ø|Δ|,Max|Δ|
Spieler (ELO Jan23),,,,,,,,,,,,,,,,,,,,,
Garv Rai (2181),47.8,-45.6,-21.6,20.8,418.4,-418.40,-54.6,-1.38,-1.96,181.55,...,-40.20,39.78,0.36,-27.0,27.36,0.32,0.0,1561.0,67.9,418.4
Harsh Suresh (2280),-66.2,-24.8,-0.2,-0.2,129.8,-204.40,118.5,-43.80,142.66,-140.40,...,103.60,-91.93,-1.10,5.3,4.00,5.00,-3.7,1522.2,66.2,226.4
Mayank Chakraborty (2179),NaN,NaN,NaN,NaN,NaN,-143.60,28.6,-39.28,72.88,-52.45,...,300.52,-179.40,-7.10,-13.8,6.10,11.50,26.7,1185.7,65.9,300.5
"Bodrogi, Bendeguz (2073)",NaN,NaN,78.2,-221.6,77.0,0.00,91.2,-45.60,-31.20,10.18,...,66.40,-117.80,28.40,-13.4,2.70,-1.40,-1.9,1132.4,53.9,221.6
"Piliposyan, Robert (2295)",-0.2,48.4,-48.2,47.6,-46.2,14.72,-40.0,86.00,-92.60,-30.20,...,-0.40,-0.60,1.20,-23.1,21.60,-21.50,22.0,1029.4,44.8,163.8
"Collin, Moritz Valentin (2123)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.60,-94.40,...,36.80,-17.04,-56.92,75.2,-23.40,32.60,24.2,1020.2,68.0,163.9
"Davtyan, Arsen (2293)",NaN,NaN,38.4,-5.0,58.2,-107.86,114.2,-4.60,-15.60,-168.45,...,0.20,0.20,-5.60,-19.5,43.20,-6.60,-12.0,991.2,47.2,168.4
Aarav Dengla (2357),NaN,NaN,NaN,0.0,0.0,1.40,-91.4,93.58,-0.88,-66.22,...,-19.60,0.60,-14.00,7.8,120.40,-113.70,-0.1,972.4,48.6,178.9
"Ljepic, Andrej (2131)",NaN,NaN,NaN,NaN,30.2,11.20,188.8,-188.20,-71.40,71.11,...,6.80,-6.40,-0.20,8.4,-45.00,37.70,14.6,927.2,48.8,188.8


---
## 4 — ELO < 2000

In [24]:
print('=' * 70)
print('ELO < 2000  (gemessen Jan 2023)')
print('=' * 70)
sub = df[df['elo_group'] == 'ELO < 2000']
print(f'{sub["fide_id"].nunique()} Spieler, {len(sub):,} Fenster')
pct_low = pivot_table(sub, 'ELO < 2000')
display(style_pivot(pct_low))
top10_table(sub, 'ELO < 2000')


ELO < 2000  (gemessen Jan 2023)
105 Spieler, 1,792 Fenster


period,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,20-01,20-07,21-01,21-07,22-01,22-07,23-01,23-07,24-01,24-07,25-01,25-07,26-01
|delta| ELO < 2000,,,,,,,,,,,,,,,,,,,,,,,
=0,22.2,30.6,29.4,35.2,32.1,31.0,37.1,29.2,30.4,25.0,30.4,58.2,77.4,34.9,25.3,20.9,18.6,19.4,0.0,19.0,9.8,8.6,8.6
1-2,15.6,22.4,25.5,20.4,19.6,24.1,35.5,32.3,20.3,26.4,31.6,32.9,11.9,36.0,17.2,19.8,17.5,20.4,2.0,34.0,25.5,22.9,14.3
3-5,11.1,10.2,9.8,11.1,7.1,5.2,6.5,10.8,8.7,9.7,7.6,0.0,0.0,2.3,3.4,7.7,11.3,7.1,8.2,9.0,9.8,10.5,13.3
--- ok ≤5 ---,48.9,63.2,64.7,66.7,58.8,60.3,79.1,72.3,59.4,61.1,69.6,91.1,89.3,73.2,45.9,48.4,47.4,46.9,10.2,62.0,45.1,42.0,36.2
6-10,8.9,12.2,9.8,7.4,3.6,6.9,4.8,7.7,7.2,8.3,13.9,5.1,4.8,14.0,10.3,12.1,14.4,13.3,3.1,18.0,16.7,16.2,14.3
11-20,22.2,10.2,15.7,13.0,21.4,15.5,4.8,9.2,8.7,4.2,7.6,3.8,1.2,5.8,14.9,8.8,13.4,17.3,16.3,6.0,10.8,13.3,25.7
21-50,15.6,12.2,9.8,11.1,12.5,15.5,9.7,10.8,17.4,15.3,7.6,0.0,3.6,3.5,13.8,17.6,18.6,15.3,24.5,10.0,18.6,17.1,21.0
>50,4.4,2.0,0.0,1.9,3.6,1.7,1.6,0.0,7.2,11.1,1.3,0.0,1.2,3.5,14.9,13.2,6.2,7.1,45.9,4.0,8.8,11.4,2.9
N (Fenster),45.0,49.0,51.0,54.0,56.0,58.0,62.0,65.0,69.0,72.0,79.0,79.0,84.0,86.0,87.0,91.0,97.0,98.0,98.0,100.0,102.0,105.0,105.0



--- ELO < 2000: Top-10 nach Σ|Δ| (delta = erwartet − gescraped, +rot / -blau) ---


,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,20-01,20-07,21-01,21-07,22-01,22-07,23-01,23-07,24-01,24-07,25-01,25-07,26-01,Σ|Δ|,Ø|Δ|,Max|Δ|
Spieler (ELO Jan23),,,,,,,,,,,,,,,,,,,,,,,,,,
"Guba, Viktor (1834)",,,,,,-0.6,+0.0,-0.2,+182.6,-181.6,+0.0,+0.0,+0.0,+0.2,-0.4,-0.8,-76.0,+76.8,+69.2,+6.0,+63.2,-57.0,-25.8,740.4,41.1,182.6
"Radzimski, Antoni (1734)",,,,,,,,,,,,,+9.8,+97.0,-104.8,-6.5,-6.3,+24.0,-202.9,+185.6,+14.4,+34.4,-53.8,739.4,67.2,202.9
"Kornienko, Konstantin (1896)",,,,,,,,,,-0.4,+0.2,+0.0,+144.0,-144.0,-0.8,+0.0,+28.6,-27.8,+109.4,-59.6,-23.2,+31.6,-30.5,600.1,42.9,144.0
"Chiriac, Daniel (1949)",-14.2,-32.8,-15.0,-35.2,+145.0,-79.8,+4.0,-32.2,+41.4,+50.1,-59.2,+0.0,+0.0,+0.0,+13.6,-14.0,+0.0,-2.6,+15.6,+1.8,-2.4,+0.2,+10.4,569.5,24.8,145.0
"Boschetti, Claudio (1988)",+40.0,-31.8,+15.2,-12.2,+21.6,-23.8,+1.0,-10.4,+5.0,-27.8,+20.4,+17.0,-30.4,+73.4,-10.4,-25.4,-3.4,-17.0,+30.0,-33.6,+41.0,-62.0,+15.8,568.6,24.7,73.4
"Uhlmann, Tim (1745)",,,,,,,,,,+0.0,-0.6,+0.2,+0.0,-1.0,-112.8,+114.0,+71.6,-73.4,+41.2,-9.4,-23.8,+52.8,-18.4,519.2,37.1,114.0
"Merk, Daniel (1965)",-196.6,+155.2,-42.0,+0.0,+44.8,-45.2,+0.0,+0.0,+0.0,+0.0,+0.2,+0.0,+0.0,+0.0,+0.0,+0.0,+0.0,+0.0,+14.0,+0.0,+0.0,+0.0,-9.0,507.0,22.0,196.6
"Madhavan, Kailash (1468)",,,,,,,,,,,,,,-1.6,+21.8,-20.2,+36.2,-36.4,+161.0,+0.2,-96.4,+96.2,-32.4,502.4,50.2,161.0
"Ogur, Nidal Hevi (1511)",,,,,,,+0.6,+0.0,+37.0,-60.6,+23.0,+0.0,+0.0,+0.8,+58.2,-58.0,-39.8,+37.8,+144.6,+0.0,+9.4,-9.8,+20.0,499.6,29.4,144.6


,15-02,15-07,16-01,16-07,17-01,17-07,18-01,18-07,19-01,19-07,...,23-01,23-07,24-01,24-07,25-01,25-07,26-01,Σ|Δ|,Ø|Δ|,Max|Δ|
Spieler (ELO Jan23),,,,,,,,,,,,,,,,,,,,,
"Guba, Viktor (1834)",NaN,NaN,NaN,NaN,NaN,-0.6,0.0,-0.2,182.6,-181.60,...,-76.00,76.8,69.20,6.00,63.2,-57.00,-25.80,740.4,41.1,182.6
"Radzimski, Antoni (1734)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-6.28,24.0,-202.92,185.55,14.4,34.40,-53.80,739.4,67.2,202.9
"Kornienko, Konstantin (1896)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.40,...,28.60,-27.8,109.40,-59.60,-23.2,31.64,-30.50,600.1,42.9,144.0
"Chiriac, Daniel (1949)",-14.2,-32.8,-15.0,-35.2,145.0,-79.8,4.0,-32.2,41.4,50.08,...,0.00,-2.6,15.60,1.80,-2.4,0.20,10.40,569.5,24.8,145.0
"Boschetti, Claudio (1988)",40.0,-31.8,15.2,-12.2,21.6,-23.8,1.0,-10.4,5.0,-27.80,...,-3.40,-17.0,30.00,-33.60,41.0,-62.00,15.80,568.6,24.7,73.4
"Uhlmann, Tim (1745)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,...,71.60,-73.4,41.20,-9.40,-23.8,52.80,-18.40,519.2,37.1,114.0
"Merk, Daniel (1965)",-196.6,155.2,-42.0,0.0,44.8,-45.2,0.0,0.0,0.0,0.00,...,0.00,0.0,14.00,0.00,0.0,0.00,-9.00,507.0,22.0,196.6
"Madhavan, Kailash (1468)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,36.20,-36.4,161.00,0.20,-96.4,96.20,-32.40,502.4,50.2,161.0
"Ogur, Nidal Hevi (1511)",NaN,NaN,NaN,NaN,NaN,NaN,0.6,0.0,37.0,-60.60,...,-39.80,37.8,144.60,0.00,9.4,-9.80,20.00,499.6,29.4,144.6
